# 9.2 Data in lists

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

For an unindexed (scalar) parameter, a `data` statement assigns one value:

```ampl
param avail := 40;
```

Most of a typical model's parameters are indexed over sets, however, and their values are
specified in a variety of lists and tables that are introduced in this section and the next,
respectively.

We start with sets of simple one-dimensional objects, and the one-dimensional collections
of parameters indexed over them. We then turn to two-dimensional sets and parameters,
for which we have the additional option of organizing the `data` into "slices". The
options for two dimensions are then shown to generalize readily to higher dimensions, for
which we present some three-dimensional examples. Finally, we show how `data` statements
for a set and the parameters indexed over it can be combined to provide a more
concise and convenient representation.

Lists of one-dimensional sets and parameters
For a parameter indexed over a one-dimensional set like

```ampl
set PROD;
param rate {PROD} > 0;
```

the specification of the set can be simply a listing of its members:

```ampl
set PROD := bands coils plate ;
```

and the parameter's specification may be virtually the same except for the addition of a
value after each set member:

```ampl
param rate := bands 200            coils 140      plate 160 ;
```

The parameter specification could equally well be written

```ampl
param rate :=
        bands       200
        coils       140
        plate       160 ;
```

since extra spaces and line breaks are ignored.

If a one-dimensional set has been declared with the attribute ordered or
circular ([Section 5.6](../05/5_6_ordered_sets.ipynb)), then the ordering of its members is taken from the `data` statement
that defines it. For example, we specified

```ampl
set WEEKS := 27sep 04oct 11oct 18oct ;
```

as the membership of the ordered set WEEKS in [Figure 5-4](../05/5_6_ordered_sets.ipynb#fig-5-4).

Members of a set must all be different; AMPL will warn of duplicates:

```ampl
duplicate member coils for set PROD
context: set PROD := bands coils plate coils >>> ; <<<
```

Also a parameter may not be given more than one value for each member of the set over
which it is indexed. A violation of this rule provokes a similar message:

```ampl
rate['bands'] already defined
context: param rate := bands 200 bands 160 >>> ; <<<
```

The context bracketed by >>> and <<< isn't the exact point of the error, but the message
makes the situation clear.

A set may be specified as empty by giving an empty list of members; simply put the
semicolon right after the := operator. A parameter indexed over an empty set has no `data`
associated with it.

Lists of two-dimensional sets and parameters
The extension of `data` lists to the two-dimensional case is largely straightforward, but
with each set member denoted by a pair of objects. As an example, consider the following
sets from [Figure 6-2a](../06/6_2_subsets_and_slices_of_ordered_pairs.ipynb#fig-6-2a):

```ampl
set ORIG;       # origins
set DEST;       # destinations
set LINKS within {ORIG,DEST}; # transportation links
```

The members of ORIG and DEST can be given as for any one-dimensional sets:

```ampl
set ORIG := GARY CLEV PITT ;
set DEST := FRA DET LAN WIN STL FRE LAF ;
```

Then the membership of LINKS may be specified as a list of tuples such as you would
find in a model's indexing expressions,

```ampl
set LINKS :=
  (GARY,DET) (GARY,LAN) (GARY,STL) (GARY,LAF) (CLEV,FRA)
  (CLEV,DET) (CLEV,LAN) (CLEV,WIN) (CLEV,STL) (CLEV,LAF)
  (PITT,FRA) (PITT,WIN) (PITT,STL) (PITT,FRE) ;
```

or as a list of pairs, without the parentheses and commas:

```ampl
set LINKS :=
  GARY DET   GARY   LAN   GARY STL   GARY LAF
  CLEV FRA   CLEV   DET   CLEV LAN   CLEV WIN
  CLEV STL   CLEV   LAF   PITT FRA   PITT WIN
  PITT STL   PITT   FRE ;
```

The order of members within each pair is significant — the first must be from ORIG, and
the second from DEST — but the pairs themselves may appear in any order.

An alternative, more concise way to describe this set of pairs is to list all second components
that go with each first component:

```ampl
set LINKS :=
  (GARY,*) DET LAN STL LAF
  (CLEV,*) FRA DET LAN WIN STL LAF
  (PITT,*) FRA WIN STL FRE ;
```

It is also easy to list all first components that go with each second component:

```ampl
set LINKS :=
  (*,FRA) CLEV PITT   (*,DET) GARY CLEV  (*,LAN) GARY CLEV
  (*,WIN) CLEV PITT   (*,LAF) GARY CLEV  (*,FRE) PITT
  (*,STL) GARY CLEV PITT ;
```

An expression such as `(GARY,*)` or `(*,FRA)`, resembling a pair but with a component
replaced by a \*, is a `data` template. Each template is followed by a list, whose entries are
substituted for the * to generate pairs; these pairs together make up a slice through the
dimension of the set where the * appears. A tuple without any \*'s, like `(GARY,DET)`,
is in effect a template that specifies only itself, so it is not followed by any values. At the
other extreme, in the table that consists of pairs alone,

```ampl
set LINKS :=
  GARY DET   GARY   LAN   GARY STL  GARY LAF
  CLEV FRA   CLEV   DET   CLEV LAN  CLEV WIN
  CLEV STL   CLEV   LAF   PITT FRA  PITT WIN
  PITT STL   PITT   FRE ;
```

a default template `(*,*)` applies to all entries.

For a parameter indexed over a two-dimensional set, the AMPL list formats are again
derived from those for sets by placing parameter values after the set members. Thus if we
have the parameter cost indexed over the set LINKS:

```ampl
param cost {LINKS} >= 0;
```

then the set `data` statement for LINKS is extended to become the following `param` `data`
statement for cost:

```ampl
param cost :=
	  GARY DET 14  GARY   LAN 11 GARY STL 16  GARY LAF 8
	  CLEV FRA 27  CLEV   DET 9 CLEV LAN 12   CLEV WIN 9
	  CLEV STL 26  CLEV   LAF 17 PITT FRA 24  PITT WIN 13
	  PITT STL 28  PITT   FRE 99 ;
```

Lists of slices through a set extend similarly, by placing a parameter value after each
implied set member. Thus, corresponding to our concise `data` statement for LINKS:

```ampl
set LINKS :=
  (GARY,*) DET LAN STL LAF
  (CLEV,*) FRA DET LAN WIN STL LAF
  (PITT,*) FRA WIN STL FRE ;
```

there is the following statement for the values of cost:

```ampl
param cost :=
  [GARY,*] DET 14  LAN 11  STL 16  LAF 8
  [CLEV,*] FRA 27  DET 9   LAN 12  WIN 9 STL 26  LAF 17
  [PITT,*] FRA 24  WIN 13  STL 28  FRE 99 ;
```

The templates are given in brackets to distinguish them from the set templates in parentheses,
but they work in the same way. Thus a template such as `[GARY,*]` indicates
that the ensuing entries will be for values of cost that have a first index of GARY, and an
entry such as DET 14 gives cost["GARY","DET"] a value of 14.

All of the above applies just as well to the use of templates that slice on the first
dimension, so that for instance you could also specify parameter cost by:

```ampl
param cost     :=
   [*,FRA]     CLEV   27   PITT   24
   [*,DET]     GARY   14   CLEV    9
   [*,LAN]     GARY   11   CLEV   12
   [*,WIN]     CLEV    9   PITT   13
   [*,STL]     GARY   16   CLEV   26   PITT 28
   [*,FRE]     PITT   99
   [*,LAF]     GARY    8   CLEV 17
```

You can even think of the list-of-pairs example,

```ampl
param cost :=
   GARY DET 14   GARY LAN 11  GARY STL 16  GARY LAF 8
   ...
```

as also being a case of this form, corresponding to the default template `[*,*]`.

Lists of higher-dimensional sets and parameters
The concepts underlying `data` lists for two-dimensional sets and parameters extend
straightforwardly to higher-dimensional cases. The only difference of any note is that
nontrivial slices may be made through more than one dimension. Hence we confine the
presentation here to some illustrative examples in three dimensions, followed by a sketch
of the general rules for the AMPL `data` list format that are given in [Section A.12](#missing) <!--- xTODO missing reference --->.

We take our example from [Section 6.3](../06/6_3_sets_of_longer_tuples.ipynb), where we suggest a version of the multicommodity
transportation `model` that defines a set of triples and costs indexed over them:

```ampl
set ROUTES within {ORIG,DEST,PROD};
       param cost {ROUTES} >= 0;
```

Suppose that ORIG and DEST are as above, that PROD only has members bands and
coils, and that ROUTES has as members certain triples from `{ORIG,DEST,PROD}`.
Then the membership of ROUTES can be given most simply by a list of triples, either

```ampl
set ROUTES :=
   (GARY,LAN,coils)   (GARY,STL,coils)   (GARY,LAF,coils)
   (CLEV,FRA,bands)   (CLEV,FRA,coils)   (CLEV,DET,bands)
   (CLEV,DET,coils)   (CLEV,LAN,bands)   (CLEV,LAN,coils)
   (CLEV,WIN,coils)   (CLEV,STL,bands)   (CLEV,STL,coils)
   (CLEV,LAF,bands)   (PITT,FRA,bands)   (PITT,WIN,bands)
   (PITT,STL,bands)   (PITT,FRE,bands)   (PITT,FRE,coils) ;
```

or

```ampl
set ROUTES :=
   GARY LAN coils   GARY   STL   coils     GARY   LAF   coils
   CLEV FRA bands   CLEV   FRA   coils     CLEV   DET   bands
   CLEV DET coils   CLEV   LAN   bands     CLEV   LAN   coils
   CLEV WIN coils   CLEV   STL   bands     CLEV   STL   coils
   CLEV LAF bands   PITT   FRA   bands     PITT   WIN   bands
   PITT STL bands   PITT   FRE   bands     PITT   FRE   coils ;
```

Using templates as before, but with three items in each template, we can break the specification
into slices through one dimension by placing one * in each template. In the following
example, we slice through the second dimension:

```ampl
set ROUTES :=
   (CLEV,*,bands) FRA DET LAN STL LAF
   (PITT,*,bands) FRA WIN STL FRE
       (GARY,*,coils) LAN STL LAF
       (CLEV,*,coils) FRA DET LAN WIN STL
       (PITT,*,coils) FRE ;
```

Because the set contains no members with origin GARY and product bands, the template
`(GARY,*,bands)` is omitted.

When the set's dimension is more than two, the slices can also be through more than
one dimension. A slice through two dimensions, in particular, naturally involves placing
two \*'s in each template. Here we slice through both the first and third dimensions:

```ampl
set ROUTES :=
   (*,FRA,*)  CLEV   bands    CLEV   coils PITT bands
   (*,DET,*)  CLEV   bands    CLEV   coils
   (*,LAN,*)  GARY   coils    CLEV   bands CLEV coils
   (*,WIN,*)  CLEV   coils    PITT   bands
   (*,STL,*)  GARY   coils    CLEV   bands CLEV coils  PITT bands
   (*,FRE,*)  PITT   bands    PITT   coils
   (*,LAF,*)  GARY   coils    CLEV   bands ;
```

Since these templates have two \*'s, they must be followed by pairs of components, which
are substituted from left to right to generate the set members. For instance the template
`(*,FRA,*)` followed by CLEV bands specifies that `(CLEV,FRA,bands)` is a member
of the set.
Any of the above forms suffices for giving the values of parameter cost as well. We
could write

```ampl
param cost :=
   [CLEV,*,bands] FRA 27   DET 9  LAN 12  STL 26  LAF 17
   [PITT,*,bands] FRA 24   WIN 13 STL 28  FRE 99
    [GARY,*,coils] LAN 11 STL 16  LAF 8
    [CLEV,*,coils] FRA 23 DET 8   LAN 10  WIN  9  STL 21
    [PITT,*,coils] FRE 81 ;
```

or

```ampl
param cost :=
   [*,*,bands]   CLEV FRA 27    CLEV DET 9     CLEV LAN 12
                 CLEV STL 26    CLEV LAF 17    PITT FRA 24
                 PITT WIN 13    PITT STL 28    PITT FRE 99
    [*,*,coils]  GARY LAN 11    GARY STL 16    GARY LAF 8
                 CLEV FRA 23    CLEV DET 8     CLEV LAN 10
                 CLEV WIN 9     CLEV STL 21    PITT FRE 81
```

or

```ampl
param cost :=
   CLEV DET bands   9    CLEV   DET   coils    8    CLEV   FRA   bands    27
   CLEV FRA coils  23    CLEV   LAF   bands   17    CLEV   LAN   bands    12
   CLEV LAN coils  10    CLEV   STL   bands   26    CLEV   STL   coils    21
   CLEV WIN coils   9    GARY   LAF   coils    8    GARY   LAN   coils    11
   GARY STL coils  16    PITT   FRA   bands   24    PITT   FRE   bands    99
   PITT FRE coils  81    PITT   STL   bands   28    PITT   WIN   bands    13 ;
```

By placing the \*'s in different positions within the templates, we can slice onedimensionally
in any of three different ways, or two-dimensionally in any of three different
ways. (The template `[*,*,*]` would specify a three-dimensional list like

```ampl
param cost :=
  CLEV DET bands  9   CLEV DET coils  8   CLEV FRA bands 27
  ...
```

as already shown above.)
More generally, a template for an n-dimensional set or parameter in list form must
have n entries. Each entry is either a legal set member or a \*. Templates for sets are
enclosed in parentheses (like the tuples in set-expressions) and templates for parameters
are enclosed in brackets (like the subscripts of parameters). Following a template is a
series of items, each item consisting of one set member for each \*, and additionally one
parameter value in the case of a parameter template. Each item defines an n-tuple, by
substituting its set members for the \*s in the template; either this tuple is added to the set
being specified, or the parameter indexed by this tuple is assigned the value in the item.

A template applies to all items between it and the next template (or the end of the `data`
statement). Templates having different numbers of \*s may even be used together in the
same `data` statement, so long as each parameter is assigned a value only once. Where no
template appears, a template of all \*s is assumed.

Combined lists of sets and parameters

When we give `data` statements for a set and a parameter indexed over it, like

```ampl
set PROD := bands coils plate ;
param rate := bands 200 coils 140                plate 160 ;
```

we are specifying the set's members twice. AMPL lets us avoid this duplication by
including the set's name in the `param` `data` statement:

```ampl
param: PROD: rate := bands 200              coils 140      plate 160 ;
```

AMPL uses this statement to determine both the membership of PROD and the values of
rate.

Another common redundancy occurs when we need to supply `data` for several parameters
indexed over the same set, such as rate, profit and market all indexed over
PROD in [Figure 1-4a](../01/tut_1_4.ipynb#fig-1-4a). Rather than write a separate `data` statement for each parameter,

```ampl
param rate   := bands 200    coils 140       plate 160 ;
param profit := bands   25   coils   30      plate   29 ;
param market := bands 6000   coils 4000      plate 3500 ;
```

we can combine these statements into one by listing all three parameter names after the
keyword `param`:

```ampl
param: rate profit market :=
  bands 200 25 6000 coils 140 30 4000                 plate 160 29 3500 ;
```

Since AMPL ignores extra spaces and line breaks, we have the option of rearranging this
information into an easier-to-read table:

```ampl
param:       rate     profit     market :=
  bands       200       25        6000
  coils       140       30        4000
  plate       160       29        3500 ;
```

Either way, we still have the option of adding the indexing set's name to the statement,

```ampl
param: PROD:  rate    profit     market :=
       bands   200      25        6000
       coils   140      30        4000
       plate   160      29        3500 ;
```

so that the specifications of the set and all three parameters are combined.

The same rules apply to lists of any higher-dimensional sets and the parameters
indexed over them. Thus for our two-dimensional example LINKS we could write

```ampl
param: LINKS: cost :=
   GARY DET 14 GARY LAN 11 GARY STL 16   GARY LAF 8
   CLEV FRA 27 CLEV DET 9 CLEV LAN 12    CLEV WIN 9
   CLEV STL 26 CLEV LAF 17 PITT FRA 24   PITT WIN 13
   PITT STL 28 PITT FRE 99 ;
```

to specify the membership of LINKS and the values of the parameter cost indexed over
it, or

```ampl
param:    LINKS: cost   limit :=
  GARY    DET     14     1000
  GARY    LAN     11      800
  GARY    STL     16     1200
  GARY    LAF      8     1100
  CLEV    FRA     27     1200
  CLEV    DET      9      600
  CLEV    LAN     12      900
  CLEV    WIN      9      950
  CLEV    STL     26     1000
  CLEV    LAF     17      800
  PITT    FRA     24     1500
  PITT    WIN     13     1400
  PITT    STL     28     1500
  PITT    FRE     99     1200 ;
```

to specify the values of cost and limit together. The same options apply when templates
are used, making possible further alternatives such as

```ampl
param: LINKS: cost :=
   [GARY,*] DET 14 LAN 11  STL 16    LAF 8
   [CLEV,*] FRA 27 DET 9   LAN 12    WIN 9 STL 26 LAF 17
   [PITT,*] FRA 24 WIN 13  STL 28    FRE 99 ;
```

and

```ampl
param:   LINKS:  cost   limit :=
  [GARY,*] DET    14     1000
           LAN    11      800
           STL    16     1200
           LAF     8     1100
  [CLEV,*] FRA    27     1200
           DET     9      600
           LAN    12      900
           WIN     9      950
           STL    26     1000
           LAF    17      800
  [PITT,*] FRA    24     1500
           WIN    13     1400
           STL    28     1500
           FRE    99     1200 ;
```

Here the membership of the indexing set is specified along with the two parameters; for
example, the template `[GARY,*]` followed by the set member DET and the values 14
and 1000 indicates that `(GARY,DET)` is to be added to the set LINKS, that
cost[GARY,DET] has the value 14, and that limit[GARY,DET] has the value 1000.

As our illustrations suggest, the key to the interpretation of a `param` statement that
provides values for several parameters or for a set and parameters is in the first line,
which consists of `param` followed by a colon, then optionally the name of an indexing
set followed by a colon, then by a list of parameter names terminated by the := assignment
operator. Each subsequent item in the list consists of a number of set members
equal to the number of \*s in the most recent template and then a number of parameter
values equal to the number of parameters listed in the first line.

Normally the parameters listed in the first line of a `param` statement are all indexed
over the same set. This need not be the case, however, as seen in the case of [Figure 5-1](../05/5_3_set_operations.ipynb#fig-5-1).
For this variation on the diet `model`, the nutrient restrictions are given by

```ampl
set MINREQ;
set MAXREQ;
param n_min {MINREQ} >= 0;
param n_max {MAXREQ} >= 0;
```

so that n_min and n_max are indexed over sets of nutrients that may overlap but that are
not likely to be the same.

Our sample `data` for this `model` specifies:

```ampl
set MINREQ := A B1 B2 C CAL ;
set MAXREQ := A NA CAL ;
param:     n_min     n_max :=
   A        700      20000
   C        700          .
   B1         0          .
   B2         0          .
   NA         .      50000
   CAL    16000      24000 ;
```

Each period or dot (.) indicates to AMPL that no value is being given for the corresponding
parameter and index. For example, since MINREQ does not contain a member NA,
the parameter `n_min[NA]` is not defined; consequently a . is given as the entry for NA
and n_min in the `data` statement. We cannot simply leave a space for this entry, because
AMPL will take it to be 50000: `data` mode processing ignores all extra spaces. Nor
should we put a zero in this entry; in that case we will get a message like

```ampl
error processing param n_min:
        invalid subscript n_min['NA'] discarded.
```

when AMPL first tries to access n_min, usually at the first `solve`.

When we name a set in the first line of a `param` statement, the set must not yet have a
value. If the specification of parameter `data` in [Figure 5-1](../05/5_3_set_operations.ipynb#fig-5-1) had been given as

```ampl
param: NUTR: n_min    n_max :=
       A       700    20000
       C       700        .
       B1        0        .
       B2        0        .
       NA        .    50000
       CAL   16000    24000 ;
```

AMPL would have generated the error message

```ampl
dietu.dat, line 16 (offset 366):
        NUTR was defined in the model
context: param: NUTR >>> : <<<    n_min  n_max :=
```

because the declaration of NUTR in the `model`,

```ampl
set NUTR = MINREQ union MAXREQ;
```

defines it already as the union of MINREQ and MAXREQ.